# PFS AG Log Explorer

Interactive exploration of PFS Auto-Guider actor log files.

**Run all cells** to launch the app, or serve it with:
```
panel serve ag_explorer.ipynb --show
```

Controls:
- **Log directory** — type any path; file list refreshes automatically
- **Log file** — newest first, with file sizes shown
- **Design** — filter to a single design window
- **Panels** — toggle individual plot panels on/off
- **Follow / Tail** — stream the file live; set the refresh interval in seconds


In [ ]:
import importlib.util
import sys
from bisect import bisect_right
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path

import panel as pn
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pn.extension('plotly', sizing_mode='stretch_width')


In [ ]:
# Import shared helpers (parsers, DataStore, time utils) from ag_common.py
from ag_common import (
    DataStore,
    HST, _night_bounds_from_store,
    parse_line,
)


In [ ]:
# Define notebook directory path for helpers that still reference _HERE
# (backward-compat with earlier cells).
_HERE = Path().resolve()


In [ ]:
_record_cache: dict[str, dict] = {}


def _fmt_size(n: int) -> str:
    if n < 1024:       return f'{n} B'
    if n < 1_048_576:  return f'{n/1024:.1f} KB'
    return f'{n/1_048_576:.1f} MB'


def _parse_log(path: str) -> dict[str, list]:
    """Parse entire file, caching results so re-selecting is instant."""
    if path not in _record_cache:
        state: dict = {}
        records: dict[str, list] = defaultdict(list)
        with open(path, errors='replace') as f:
            for line in f:
                for rtype, rec in parse_line(line.rstrip('\n'), state):
                    records[rtype].append(rec)
        _record_cache[path] = dict(records)
    return _record_cache[path]


def _parse_from_pos(path: str, pos: int, state: dict) -> tuple[dict[str, list], int]:
    """Read from byte offset *pos*, return (new_records, new_pos)."""
    new_records: dict[str, list] = defaultdict(list)
    with open(path, errors='replace') as f:
        f.seek(pos)
        for line in f:
            for rtype, rec in parse_line(line.rstrip('\n'), state):
                new_records[rtype].append(rec)
        new_pos = f.tell()
    return dict(new_records), new_pos


def _make_store(records: dict[str, list], t_start=None, t_end=None) -> DataStore:
    store = DataStore(window_minutes=None)
    for rtype, recs in records.items():
        for rec in recs:
            if (t_start is None or rec.t >= t_start) and \
               (t_end   is None or rec.t <  t_end):
                store.push(rtype, rec)
    return store


def _last_t(records: dict[str, list]):
    last = None
    for recs in records.values():
        if recs and (last is None or recs[-1].t > last):
            last = recs[-1].t
    return last


def _context_arrays(tel_recs: list, store: 'DataStore') -> list:
    """For each tel_axes record return [design_str, visit_str, frame_str]."""
    designs = store.snapshot('design_changes')
    visits  = store.snapshot('visit_changes')
    guides  = store.snapshot('guide')
    d_times = [r.t for r in designs]
    v_times = [r.t for r in visits]
    g_times = [r.t for r in guides]
    out = []
    for r in tel_recs:
        di = bisect_right(d_times, r.t) - 1
        vi = bisect_right(v_times, r.t) - 1
        gi = bisect_right(g_times, r.t) - 1
        out.append([
            f'd…{str(designs[di].design_id)[-8:]}' if di >= 0 else '—',
            f'v{visits[vi].visit_id}'                  if vi >= 0 else '—',
            str(guides[gi].frame_id)                     if gi >= 0 else '—',
        ])
    return out


def _guide_context_arrays(guide_recs: list, store: 'DataStore') -> list:
    """For each guide record return [design_str, visit_str, frame_str]."""
    designs = store.snapshot('design_changes')
    visits  = store.snapshot('visit_changes')
    d_times = [r.t for r in designs]
    v_times = [r.t for r in visits]
    out = []
    for r in guide_recs:
        di = bisect_right(d_times, r.t) - 1
        vi = bisect_right(v_times, r.t) - 1
        out.append([
            f'd…{str(designs[di].design_id)[-8:]}' if di >= 0 else '—',
            f'v{visits[vi].visit_id}'                  if vi >= 0 else '—',
            str(r.frame_id),
        ])
    return out


def _hst(t):
    """Convert timezone-aware datetime to naive HST for Plotly display."""
    return t.astimezone(HST).replace(tzinfo=None)


def _build_design_labels(designs: list) -> list[str]:
    labels = ['All designs']
    for i, r in enumerate(designs):
        t0 = r.t.astimezone(HST).strftime('%H:%M')
        t1_obj = designs[i + 1].t if i + 1 < len(designs) else None
        t1 = t1_obj.astimezone(HST).strftime('%H:%M') if t1_obj else 'end'
        labels.append(f'd…{str(r.design_id)[-8:]}  {t0}–{t1} HST')
    return labels


def _has_data(store: DataStore) -> bool:
    return any(store.snapshot(k) for k in ('guide', 'focus', 'star_stats', 'camera_counts', 'tel_axes'))


In [ ]:
PANEL_NAMES = ['Guide Offsets', 'InR & Scale', 'Focus', 'Camera Counts', 'Star Quality', 'Telescope']

_PANEL_TITLES = {
    'Guide Offsets':  'Guide offsets',
    'InR & Scale':    'Rotation offset & scale',
    'Focus':          'Focus offsets',
    'Star Quality':   'Star quality / seeing proxy',
    'Camera Counts':  'Per-camera object counts',
    'Telescope':      'Telescope position',
}
_GUIDE_COLORS = ['#1f77b4', '#ff7f0e', '#d62728', '#9467bd']
_Z_COLORS     = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#17becf']
_CAM_COLORS   = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#17becf']
_CAM_LABELS   = ['AG1', 'AG2', 'AG3', 'AG4', 'AG5', 'AG6']


def _event_shapes_and_annotations(store: DataStore) -> tuple[list, list]:
    shapes, annotations = [], []
    for rec in store.snapshot('visit_changes'):
        x = _hst(rec.t).isoformat()
        is_focus = getattr(rec, 'mode', 'unknown') == 'focus'
        v_color = 'seagreen' if is_focus else 'royalblue'
        v_label = f'[F]v{rec.visit_id}' if is_focus else f'v{rec.visit_id}'
        shapes.append(dict(
            type='line', xref='x', yref='paper',
            x0=x, x1=x, y0=0, y1=1,
            line=dict(color=v_color, width=1.4, dash='dot' if is_focus else 'solid'),
            opacity=0.4, layer='below',
        ))
        annotations.append(dict(
            x=x, y=0.02, xref='x', yref='paper',
            text=v_label, textangle=-90,
            showarrow=False, font=dict(size=9, color=v_color),
            xanchor='left', yanchor='bottom',
        ))
    for rec in store.snapshot('design_changes'):
        x = _hst(rec.t).isoformat()
        shapes.append(dict(
            type='line', xref='x', yref='paper',
            x0=x, x1=x, y0=0, y1=1,
            line=dict(color='darkorange', width=1.4, dash='dash'),
            opacity=0.3, layer='below',
        ))
        annotations.append(dict(
            x=x, y=0.02, xref='x', yref='paper',
            text=f'd{str(rec.design_id)[-8:]}', textangle=-90,
            showarrow=False, font=dict(size=9, color='darkorange'),
            xanchor='right', yanchor='bottom',
        ))
    for rec in store.snapshot('reconfigs'):
        x = _hst(rec.t).isoformat()
        label = ' '.join(f'{k}={v}' for k, v in rec.params.items())
        shapes.append(dict(
            type='line', xref='x', yref='paper',
            x0=x, x1=x, y0=0, y1=1,
            line=dict(color='mediumpurple', width=1.0, dash='dot'),
            opacity=0.4, layer='below',
        ))
        annotations.append(dict(
            x=x, y=0.98, xref='x', yref='paper',
            text=label, textangle=-90,
            showarrow=False, font=dict(size=8, color='mediumpurple'),
            xanchor='right', yanchor='top',
        ))
    return shapes, annotations



def _mc_step_traces(reconfigs, x_start, x_end, yaxis, legendgroup):
    """Return ±max_correction step-function traces, or [] if none found."""
    mc = sorted(
        ((r.t, float(r.params['max_correction'])) for r in reconfigs if 'max_correction' in r.params),
        key=lambda p: p[0],
    )
    if not mc:
        return []
    ts = [_hst(t) for t, _ in mc] + [x_end]
    vals = [v for _, v in mc] + [mc[-1][1]]
    traces = []
    for sign in (1, -1):
        traces.append(go.Scatter(
            x=ts, y=[sign * v for v in vals],
            name='max_correction',
            mode='lines',
            line=dict(color='rgba(220,50,50,0.5)', width=1.2, dash='dot'),
            line_shape='hv',
            legendgroup=legendgroup,
            showlegend=False,
            xaxis='x', yaxis=yaxis,
            hovertemplate=f'max_correction: %{{y:.1f}} arcsec<extra></extra>',
        ))
    return traces


def _build_scatter_column(store: 'DataStore', show_acq: bool, clamp_offsets: bool = True) -> 'go.Figure':
    """Four scatter plots in one figure — one per relevant time-series panel.

    Rows (top to bottom):
      1. dAz vs dEl         (Guide Offsets)
      2. dInR vs dScale     (InR & Scale)
      3. dFocus vs counts   (Camera Counts + Star Quality, tall)
      4. Peak ADU vs size   (Star Quality)

    Points are colored by guiding mode (rows 1/2/4) or by camera (row 3).
    """
    from plotly.subplots import make_subplots

    _MODE_COLOR = {
        'autoguide': '#2ca02c',
        'acquire':   '#ff7f0e',
        'converge':  '#9467bd',
        'focus':     'seagreen',
        'unknown':   'gray',
    }

    guide = store.snapshot('guide')
    if not show_acq:
        guide = [r for r in guide if r.mode == 'autoguide']

    ss = store.snapshot('star_stats')
    if not show_acq:
        ss = [r for r in ss if r.mode == 'autoguide']

    cam_counts = store.snapshot('camera_counts')
    if not show_acq:
        cam_counts = [r for r in cam_counts if r.mode == 'autoguide']

    # frame_id -> dfocus lookup for the dFocus vs counts scatter
    fid_to_dfocus = {r.frame_id: r.dfocus for r in guide if r.frame_id is not None}

    fig = make_subplots(
        rows=4, cols=1,
        row_heights=[1, 1, 2, 1],
        subplot_titles=['dAz vs dEl', 'dInR vs dScale', 'dFocus vs counts (per cam)', 'Peak ADU vs PSF size'],
        vertical_spacing=0.07,
    )

    modes = list(dict.fromkeys(r.mode for r in guide))

    # ── Row 1: dAz vs dEl ─────────────────────────────────────────────────────
    for mode in modes:
        for invalid in (False, True):
            pts = [r for r in guide if r.mode == mode and (r.status != 'OK') == invalid]
            if not pts:
                continue
            _col = _MODE_COLOR.get(mode, 'gray')
            fig.add_trace(go.Scatter(
                x=[r.daz for r in pts], y=[r.del_ for r in pts],
                mode='markers', name=mode if not invalid else f'{mode} ✕',
                marker=dict(size=5 if invalid else 4, color=_col, opacity=0.9 if invalid else 0.7,
                            symbol='x' if invalid else 'circle'),
                legendgroup=f'mode_{mode}', showlegend=(not invalid),
                customdata=[r.frame_id for r in pts],
                hovertemplate='dAz: %{x:.4f}<br>dEl: %{y:.4f}<br>frame: %{customdata}<extra></extra>',
            ), row=1, col=1)

    # ── Row 2: dInR vs dScale ─────────────────────────────────────────────────
    for mode in modes:
        for invalid in (False, True):
            pts = [r for r in guide if r.mode == mode and (r.status != 'OK') == invalid]
            if not pts:
                continue
            _col = _MODE_COLOR.get(mode, 'gray')
            fig.add_trace(go.Scatter(
                x=[r.dscale * 1e6 for r in pts], y=[r.dinr for r in pts],
                mode='markers', name=mode if not invalid else f'{mode} ✕',
                marker=dict(size=5 if invalid else 4, color=_col, opacity=0.9 if invalid else 0.7,
                            symbol='x' if invalid else 'circle'),
                legendgroup=f'mode_{mode}', showlegend=False,
                customdata=[r.frame_id for r in pts],
                hovertemplate='dScale: %{x:.2f}×10⁻⁶<br>dInR: %{y:.4f}<br>frame: %{customdata}<extra></extra>',
            ), row=2, col=1)

    # ── Row 3: dFocus vs per-camera counts ────────────────────────────────────
    for cam_idx in range(6):
        pts = [
            (fid_to_dfocus[r.frame_id], r.counts[cam_idx])
            for r in cam_counts
            if r.frame_id in fid_to_dfocus and cam_idx < len(r.counts)
        ]
        if pts:
            dfoc_vals, cnt_vals = zip(*pts)
            fig.add_trace(go.Scatter(
                x=list(dfoc_vals), y=list(cnt_vals),
                mode='markers', name=_CAM_LABELS[cam_idx],
                marker=dict(size=4, color=_CAM_COLORS[cam_idx], opacity=0.7),
                legendgroup=f'cam_{cam_idx}', showlegend=True,
                hovertemplate=(
                    f'{_CAM_LABELS[cam_idx]}<br>'
                    'dFocus: %{x:.3f} mm<br>Counts: %{y}<extra></extra>'
                ),
            ), row=3, col=1)

    # ── Row 4: Peak ADU vs PSF size ───────────────────────────────────────────
    ss_modes = list(dict.fromkeys(r.mode for r in ss))
    for mode in ss_modes:
        pts = [r for r in ss if r.mode == mode]
        fig.add_trace(go.Scatter(
            x=[r.peak for r in pts], y=[r.size for r in pts],
            mode='markers', name=mode,
            marker=dict(size=4, color=_MODE_COLOR.get(mode, 'gray'), opacity=0.7),
            legendgroup=f'mode_{mode}', showlegend=False,
            hovertemplate='Peak: %{x:.0f} ADU<br>Size: %{y:.2f} pix<extra></extra>',
        ), row=4, col=1)

    fig.update_layout(
        margin=dict(l=55, r=10, t=30, b=30),
        showlegend=True,
        legend=dict(orientation='v', x=1.05, y=1, font=dict(size=12)),
        plot_bgcolor='white',
        paper_bgcolor='white',
    )
    _grid = dict(showgrid=True, gridcolor='#eee', zeroline=True, zerolinecolor='#ddd')
    _r1_range = dict(range=[-0.5, 0.5]) if clamp_offsets else {}
    fig.update_xaxes(title_text='dAz (arcsec)', row=1, col=1, **_grid, **_r1_range, title_font=dict(size=10))
    fig.update_yaxes(title_text='dEl (arcsec)', row=1, col=1, **_grid, **_r1_range, title_font=dict(size=10))
    fig.update_xaxes(title_text='dScale (×10⁻⁶)', row=2, col=1, **_grid, title_font=dict(size=10))
    fig.update_yaxes(title_text='dInR (arcsec)', row=2, col=1, **_grid, title_font=dict(size=10))
    fig.update_xaxes(title_text='dFocus (mm)', row=3, col=1, **_grid, title_font=dict(size=10))
    fig.update_yaxes(title_text='Counts', row=3, col=1, **_grid, title_font=dict(size=10))
    fig.update_xaxes(title_text='Peak ADU', row=4, col=1, **_grid, title_font=dict(size=10))
    fig.update_yaxes(title_text='PSF size (pix)', row=4, col=1, **_grid, title_font=dict(size=10))

    return fig


def _build_figure(store: DataStore, active_panels: list[str], x_start, x_end) -> go.Figure:
    import math
    n = len(active_panels)
    if n == 0:
        return go.Figure()

    GAP = 0.03

    def _dom(i):
        """Y-axis domain [bottom, top] for panel i (0 = topmost panel)."""
        h = (1.0 - (n - 1) * GAP) / n
        return [round(1.0 - (i + 1) * h - i * GAP, 4),
                round(1.0 - i * h - i * GAP, 4)]

    def _yaxes(i):
        """Return (p_ref, s_ref, p_key, s_key) for panel i."""
        p = 2 * i + 1
        s = 2 * i + 2
        p_ref = 'y' if p == 1 else f'y{p}'
        s_ref = f'y{s}'
        p_key = 'yaxis' if p == 1 else f'yaxis{p}'
        s_key = f'yaxis{s}'
        return p_ref, s_ref, p_key, s_key

    fig = go.Figure()
    layout_updates = {}

    for i, panel in enumerate(active_panels):
        dom = _dom(i)
        p_ref, s_ref, p_key, s_key = _yaxes(i)
        lg = f'p{i}'

        layout_updates[p_key] = dict(domain=dom)
        layout_updates[s_key] = dict(overlaying=p_ref, side='right')

        if panel == 'Guide Offsets':
            recs = store.snapshot('guide')
            if not show_acq_frames.value:
                recs = [r for r in recs if r.mode == 'autoguide']
            if recs:
                ts  = [_hst(r.t) for r in recs]
                ctx = _guide_context_arrays(recs, store)
                invalid_idx = [j for j, r in enumerate(recs) if r.status != 'OK']
                _azel = guide_coords.value == 'Az/El'
                _pairs = (
                    [('daz', 'dAz'), ('del_', 'dEl')]
                    if _azel else
                    [('dra', 'dRA'), ('ddec', 'dDec')]
                )
                for first, ((attr, label), color) in enumerate(zip(_pairs, _GUIDE_COLORS)):
                    if first == 0:
                        _primary = 'dAz' if _azel else 'dRA'
                        ht = (
                            '<b>%{x|%H:%M:%S} HST</b><br>'
                            f'{_primary}: %{{y:.4f}} arcsec<br>'
                            '──────────────────<br>'
                            'Design: %{customdata[0]}<br>'
                            'Visit:  %{customdata[1]}<br>'
                            'Frame:  %{customdata[2]}'
                            '<extra></extra>'
                        )
                    else:
                        ht = f'{label}: %{{y:.4f}} arcsec<extra></extra>'
                    vals = [getattr(r, attr) for r in recs]
                    fig.add_trace(go.Scatter(
                        x=ts, y=vals,
                        name=label, mode='lines+markers',
                        marker=dict(size=4), line=dict(color=color, width=1.2),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                        customdata=ctx, hovertemplate=ht,
                    ))
                    if invalid_idx:
                        fig.add_trace(go.Scatter(
                            x=[ts[j] for j in invalid_idx],
                            y=[vals[j] for j in invalid_idx],
                            name='INVALID_OFFSET' if first == 0 else None,
                            mode='markers',
                            marker=dict(size=8, color=color, symbol='x', line=dict(width=2)),
                            legendgroup=lg, showlegend=(first == 0),
                            xaxis='x', yaxis=p_ref,
                            hovertemplate='%{y:.4f} arcsec — INVALID_OFFSET<extra></extra>',
                        ))
                for tr in _mc_step_traces(store.snapshot('reconfigs'), x_start, x_end, p_ref, lg):
                    fig.add_trace(tr)
            # Dashed threshold lines at ±0.125 and ±0.25 arcsec
            for y_val in (0.125, 0.25, -0.125, -0.25):
                fig.add_shape(
                    type='line', xref='paper', x0=0, x1=1,
                    yref=p_ref, y0=y_val, y1=y_val,
                    line=dict(color='gray', width=0.8, dash='dash'),
                    layer='below',
                )
            upd = dict(title_text='arcsec')
            if not full_offset_range.value:
                upd['range'] = [-0.5, 0.5]
            layout_updates[p_key].update(**upd)

        elif panel == 'InR & Scale':
            recs = store.snapshot('guide')
            if not show_acq_frames.value:
                recs = [r for r in recs if r.mode == 'autoguide']
            if recs:
                ts = [_hst(r.t) for r in recs]
                ctx = _guide_context_arrays(recs, store)
                invalid_idx = [j for j, r in enumerate(recs) if r.status != 'OK']
                dinr_vals = [r.dinr for r in recs]
                fig.add_trace(go.Scatter(
                    x=ts, y=dinr_vals,
                    name='dInR', mode='lines+markers',
                    marker=dict(size=4), line=dict(color='#2ca02c', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=p_ref,
                    customdata=ctx,
                    hovertemplate='dInR: %{y:.4f} arcsec<extra></extra>',
                ))
                if invalid_idx:
                    fig.add_trace(go.Scatter(
                        x=[ts[j] for j in invalid_idx],
                        y=[dinr_vals[j] for j in invalid_idx],
                        name='INVALID_OFFSET (InR)' if False else None,
                        mode='markers',
                        marker=dict(size=8, color='#2ca02c', symbol='x', line=dict(width=2)),
                        legendgroup=lg, showlegend=False,
                        xaxis='x', yaxis=p_ref,
                        hovertemplate='%{y:.4f} arcsec — INVALID_OFFSET<extra></extra>',
                    ))
                for tr in _mc_step_traces(store.snapshot('reconfigs'), x_start, x_end, p_ref, lg):
                    fig.add_trace(tr)
                fig.add_trace(go.Scatter(
                    x=ts, y=[r.dscale * 1e6 for r in recs],
                    name='dScale ×10⁻⁶', mode='lines+markers',
                    marker=dict(size=4, color='gray'), line=dict(color='gray', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=s_ref,
                    hovertemplate='dScale: %{y:.4f} ×10⁻⁶<extra></extra>',
                ))
            layout_updates[p_key].update(title_text='dInR (arcsec)',
                                         tickfont=dict(color='#2ca02c'),
                                         title_font=dict(color='#2ca02c'))
            layout_updates[s_key].update(title_text='dScale (×10⁻⁶)',
                                         tickfont=dict(color='gray'),
                                         title_font=dict(color='gray'))

        elif panel == 'Focus':
            g = store.snapshot('guide')
            f = store.snapshot('focus')
            if not show_acq_frames.value:
                g = [r for r in g if r.mode == 'autoguide']
                f = [r for r in f if r.mode == 'autoguide']
            if f:
                ts = [_hst(r.t) for r in f]
                for attr, color, label in zip(
                    ['z1', 'z2', 'z3', 'z4', 'z5', 'z6'],
                    _Z_COLORS,
                    ['Z1', 'Z2', 'Z3', 'Z4', 'Z5', 'Z6'],
                ):
                    vals = [getattr(r, attr) for r in f]
                    offline = all(math.isnan(v) for v in vals)
                    fig.add_trace(go.Scatter(
                        x=ts, y=vals,
                        name=f'{label} (offline)' if offline else label,
                        mode='markers',
                        marker=dict(size=4, color='lightgrey' if offline else color),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                    ))
            if g:
                ts = [_hst(r.t) for r in g]
                fig.add_trace(go.Scatter(
                    x=ts, y=[r.dfocus for r in g],
                    name='dFocus', mode='lines+markers',
                    marker=dict(size=4, color='black'), line=dict(color='black', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=p_ref,
                ))
            layout_updates[p_key].update(title_text='Focus offset (mm)')

        elif panel == 'Star Quality':
            ss = store.snapshot('star_stats')
            if not show_acq_frames.value:
                ss = [r for r in ss if r.mode == 'autoguide']
            if ss:
                import pandas as pd
                ts = [_hst(r.t) for r in ss]
                sizes = [r.size for r in ss]
                peaks = [r.peak for r in ss]
                win = max(5, len(ts) // 10)
                s_sizes = pd.Series(sizes)
                s_peaks = pd.Series(peaks)
                roll_size = s_sizes.rolling(win, center=True, min_periods=1).median().tolist()
                err_size = s_sizes.rolling(win, center=True, min_periods=1).std().tolist()
                fig.add_trace(go.Scatter(
                    x=ts, y=peaks,
                    name='peak ADU', mode='lines+markers',
                    marker=dict(size=4, color='#ff7f0e'),
                    line=dict(color='#ff7f0e', width=1),
                    legendgroup=lg, xaxis='x', yaxis=s_ref,
                    zorder=1,
                ))
                fig.add_trace(go.Scatter(
                    x=ts, y=sizes,
                    name='PSF size (px)', mode='markers',
                    marker=dict(size=4, color='#1f77b4', opacity=0.25),
                    error_y=dict(type='data', array=err_size, visible=True,
                                 color='#1f77b4', thickness=1, width=0),
                    legendgroup=lg, xaxis='x', yaxis=p_ref,
                    zorder=2,
                ))
                fig.add_trace(go.Scatter(
                    x=ts, y=roll_size,
                    name='rolling median PSF size', mode='lines', showlegend=False,
                    line=dict(color='#1f77b4', width=3),
                    legendgroup=lg, xaxis='x', yaxis=p_ref,
                    zorder=3,
                ))
            layout_updates[p_key].update(title_text='PSF size (px)',
                                         tickfont=dict(color='#1f77b4'),
                                         title_font=dict(color='#1f77b4'))
            layout_updates[s_key].update(title_text='peak ADU (log)', type='log',
                                         tickfont=dict(color='#ff7f0e'),
                                         title_font=dict(color='#ff7f0e'))

        elif panel == 'Camera Counts':
            recs = store.snapshot('camera_counts')
            if not show_acq_frames.value:
                recs = [r for r in recs if r.mode == 'autoguide']
            if recs:
                import pandas as pd
                ts = [_hst(r.t) for r in recs]
                win = max(5, len(ts) // 10)
                for j, color in enumerate(_CAM_COLORS):
                    vals = [r.counts[j] if r.counts[j] is not None else None for r in recs]
                    roll_vals = pd.Series(vals).rolling(win, center=True, min_periods=1).median().tolist()
                    fig.add_trace(go.Scatter(
                        x=ts, y=vals,
                        name=f'AGC{j + 1}', mode='markers',
                        marker=dict(size=3, color=color, opacity=0.35),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                    ))
                    fig.add_trace(go.Scatter(
                        x=ts, y=roll_vals,
                        name=f'rolling median AGC{j + 1}', mode='lines', showlegend=False,
                        line=dict(color=color, width=3),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                    ))
            layout_updates[p_key].update(title_text='detected sources')

        elif panel == 'Telescope':
            axes_recs = store.snapshot('tel_axes')
            state_recs = store.snapshot('tel_state')
            # Primary axis: Az, El (low-cadence Gen2 status)
            if axes_recs:
                ts_ax = [_hst(r.t) for r in axes_recs]
                for attr, label, color in [
                    ('az',  'Az (deg)',  '#1f77b4'),
                    ('el',  'El (deg)',  '#ff7f0e'),
                ]:
                    fig.add_trace(go.Scatter(
                        x=ts_ax, y=[getattr(r, attr) for r in axes_recs],
                        name=label, mode='lines+markers',
                        marker=dict(size=3), line=dict(color=color, width=1.2),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                    ))
            # Secondary axis: rot (Gen2, low-cadence), InR, ADC, M2 (per-frame)
            if axes_recs:
                fig.add_trace(go.Scatter(
                    x=ts_ax, y=[r.rot for r in axes_recs],
                    name='Rot/PA (deg)', mode='lines+markers',
                    marker=dict(size=3), line=dict(color='#9467bd', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=s_ref,
                    hovertemplate='Rot/PA: %{y:.3f} deg<extra></extra>',
                ))
            if state_recs:
                ts_st = [_hst(r.t) for r in state_recs]
                for attr, label, color, dash in [
                    ('inr',    'InR (deg)',    '#2ca02c', 'solid'),
                    ('adc',    'ADC (deg)',    '#d62728', 'dash'),
                    ('m2_pos3','M2 pos3 (mm)', '#8c564b', 'dot'),
                ]:
                    fig.add_trace(go.Scatter(
                        x=ts_st, y=[getattr(r, attr) for r in state_recs],
                        name=label, mode='lines+markers',
                        marker=dict(size=3), line=dict(color=color, width=1.2, dash=dash),
                        legendgroup=lg, xaxis='x', yaxis=s_ref,
                        hovertemplate=f'{label}: %{{y:.4f}}<extra></extra>',
                    ))
            layout_updates[p_key].update(title_text='Az / El (deg)')
            layout_updates[s_key].update(title_text='Rot / InR / ADC (deg) · M2 (mm)')

    # ── Align zeros on dual y-axis panels ─────────────────────────────────────
    for i, panel in enumerate(active_panels):
        p_ref, s_ref, p_key, s_key = _yaxes(i)
        if layout_updates.get(s_key, {}).get('type') == 'log':
            continue  # log scale has no zero to align
        p_traces = [t for t in fig.data if getattr(t, 'yaxis', None) == p_ref]
        s_traces = [t for t in fig.data if getattr(t, 'yaxis', None) == s_ref]
        if not p_traces or not s_traces:
            continue
        def _finite(traces):
            return [float(v) for t in traces for v in (t.y or [])
                    if v is not None and not math.isnan(float(v))]
        pv, sv = _finite(p_traces), _finite(s_traces)
        if not pv or not sv:
            continue
        lo_p, hi_p = min(pv), max(pv)
        lo_s, hi_s = min(sv), max(sv)
        # 5 % padding
        pad_p = max(abs(hi_p - lo_p) * 0.05, 1e-9)
        pad_s = max(abs(hi_s - lo_s) * 0.05, 1e-9)
        lo_p -= pad_p; hi_p += pad_p
        lo_s -= pad_s; hi_s += pad_s
        if lo_p >= 0 or hi_p <= 0 or lo_s >= 0 or hi_s <= 0:
            continue  # zero outside a range — nothing to align
        # Pick the fraction f in [0,1] where zero must sit so both datasets fit
        f_p = -lo_p / (hi_p - lo_p)
        f_s = -lo_s / (hi_s - lo_s)
        f = max(f_p, f_s)
        r = f / (1.0 - f)  # ratio |lo| / hi at aligned zero
        def _align(lo, hi):
            if r * hi >= -lo:      # negative side is the constraint
                return -r * hi, hi
            else:                   # positive side is the constraint
                return lo, -lo / r
        layout_updates[p_key]['range'] = list(_align(lo_p, hi_p))
        layout_updates[s_key]['range'] = list(_align(lo_s, hi_s))

    shapes, event_annotations = _event_shapes_and_annotations(store)

    title_annotations = [
        dict(
            x=0, y=_dom(i)[1],
            xref='paper', yref='paper',
            text=f'<b>{_PANEL_TITLES[p]}</b>',
            xanchor='left', yanchor='bottom',
            showarrow=False, font=dict(size=11),
        )
        for i, p in enumerate(active_panels)
    ]

    fig.update_layout(
        **layout_updates,
        xaxis=dict(
            domain=[0, 1],
            range=[_hst(x_start).isoformat(), _hst(x_end).isoformat()],
            title_text='Time (HST)',
            showspikes=True,
            spikemode='across',
            spikesnap='cursor',
            spikecolor='rgba(100,100,100,0.5)',
            spikethickness=1,
            spikedash='solid',
        ),
        height=max(300, 260 * n),
        hovermode='x',
        legend=dict(orientation='v', x=-0.12, y=1, xanchor='right', tracegroupgap=12),
        margin=dict(l=180, r=30, t=40, b=60),
        shapes=shapes,
        annotations=event_annotations + title_annotations,
    )
    return fig


In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────────

_LOGS_DEFAULT = str(_HERE / 'logs')

dir_input = pn.widgets.TextInput(
    name='Log directory', value=_LOGS_DEFAULT, width=280,
)
file_select = pn.widgets.Select(name='Log file', width=280)
show_all_files = pn.widgets.Checkbox(name='Show all files (incl. <1 MB)', value=False)
show_acq_frames = pn.widgets.Checkbox(name='Show non-science frames (acquire/converge/focus)', value=False)
full_offset_range = pn.widgets.Checkbox(name='Guide offsets: full y-range', value=False)
guide_coords = pn.widgets.RadioButtonGroup(
    name='Guide offset coords', options=['Az/El', 'RA/Dec'], value='Az/El',
    button_type='default', width=200,
)
design_select = pn.widgets.Select(name='Design', width=280)
panels_check = pn.widgets.CheckBoxGroup(
    name='Panels', options=PANEL_NAMES, value=PANEL_NAMES,
)
follow_toggle = pn.widgets.Toggle(
    name='▶ Follow / Tail', button_type='success', width=150,
)
interval_input = pn.widgets.IntInput(
    name='Refresh (s)', value=5, start=1, end=300, step=1, width=100,
)
tail_window_select = pn.widgets.Select(
    name='Tail window',
    options={
        'All data': 'all',
        'Last 5 min': '5m',
        'Last 10 min': '10m',
        'Last 30 min': '30m',
        'Current visit': 'visit',
        'Current design': 'design',
    },
    value='all',
    width=150,
)
status_md = pn.pane.Markdown('', width=280)
plot_pane = pn.pane.Plotly(go.Figure(), sizing_mode='stretch_width', min_height=400)
scatter_col_pane = pn.pane.Plotly(go.Figure(), sizing_mode='stretch_height', width=650)

# ── Internal state ────────────────────────────────────────────────────────────

_record_cache_local: dict[str, dict] = {}
_busy = [False]
_follow_cb = [None]
_tail_state: dict = {}
_tail_pos = [0]
_tail_records: dict[str, list] = {}


# ── Helpers ───────────────────────────────────────────────────────────────────

def _log_path():
    d = Path(dir_input.value.strip())
    v = file_select.value
    return d / v if v else None


def _refresh_file_list(directory: str):
    p = Path(directory.strip())
    if p.is_dir():
        all_paths = sorted(p.glob('*.log'), reverse=True)
        paths = all_paths if show_all_files.value else [
            pp for pp in all_paths if pp.stat().st_size >= 1_048_576
        ]
    else:
        paths = []
    opts = {f'{pp.name}  ({_fmt_size(pp.stat().st_size)})': pp.name for pp in paths}
    file_select.param.update(options=opts, value=list(opts.values())[0] if opts else None)


def _refresh_design_list(records: dict):
    designs = records.get('design_changes', [])
    design_select.param.update(
        options=_build_design_labels(designs),
        value='All designs',
    )


def _current_records():
    """Return records to plot: tail buffer when following, cache otherwise."""
    if follow_toggle.value and _tail_records:
        return _tail_records
    p = _log_path()
    return _parse_log(str(p)) if p and p.exists() else {}


def _tail_window_bounds(records: dict):
    """Return (t_start, t_end) for the active tail window, or (None, None) for all data."""
    window = tail_window_select.value
    if window == '5m':
        return datetime.now(tz=HST) - timedelta(minutes=5), None
    if window == '10m':
        return datetime.now(tz=HST) - timedelta(minutes=10), None
    if window == '30m':
        return datetime.now(tz=HST) - timedelta(minutes=30), None
    if window == 'visit':
        visits = records.get('visit_changes', [])
        return (visits[-1].t, None) if visits else (None, None)
    if window == 'design':
        designs = records.get('design_changes', [])
        return (designs[-1].t, None) if designs else (None, None)
    return None, None  # 'all'


_frame_ts: dict[int, object] = {}  # frame_id -> datetime, updated each render


def _on_scatter_click(event):
    """Draw a vertical marker on the time-series at the clicked frame."""
    cd = event.new
    if not cd or 'points' not in cd or not cd['points']:
        return
    frame_id = cd['points'][0].get('customdata')
    if frame_id is None or frame_id not in _frame_ts:
        return
    ts = _hst(_frame_ts[frame_id]).isoformat()
    fig = plot_pane.object
    if fig is None:
        return
    shapes = [s for s in (fig.layout.shapes or []) if getattr(s, 'name', None) != 'click_marker']
    shapes = list(shapes) + [dict(
        name='click_marker',
        type='line', xref='x', yref='paper',
        x0=ts, x1=ts, y0=0, y1=1,
        line=dict(color='crimson', width=2, dash='dash'),
        opacity=0.8, layer='above',
    )]
    fig.update_layout(shapes=shapes)
    plot_pane.param.trigger('object')


scatter_col_pane.param.watch(_on_scatter_click, 'click_data')


def _render():
    p = _log_path()
    if not p:
        plot_pane.object = go.Figure()
        scatter_col_pane.object = go.Figure()
        status_md.object = '⚠️ No log file selected.'
        return
    try:
        records = _current_records()
        designs = records.get('design_changes', [])
        labels  = list(design_select.options) or ['All designs']
        sel     = design_select.value or 'All designs'

        if follow_toggle.value and tail_window_select.value != 'all':
            t_start, t_end = _tail_window_bounds(records)
            if t_start is not None:
                store = _make_store(records, t_start, t_end)
                x_start = t_start
                x_end = t_end or _last_t(records) or t_start
            else:
                store = _make_store(records)
                x_start, x_end = _night_bounds_from_store(store)
        elif sel == 'All designs':
            store = _make_store(records)
            x_start, x_end = _night_bounds_from_store(store)
        else:
            d_idx   = max(0, labels.index(sel) - 1)
            t_start = designs[d_idx].t
            t_end   = designs[d_idx + 1].t if d_idx + 1 < len(designs) else None
            store   = _make_store(records, t_start, t_end)
            x_start = t_start
            x_end   = t_end or _last_t(records) or t_start

        if not _has_data(store):
            plot_pane.object = go.Figure()
            scatter_col_pane.object = go.Figure()
            total = sum(len(v) for v in records.values())
            if total == 0:
                status_md.object = f'⚠️ **{p.name}** — empty or unrecognised format.'
            else:
                status_md.object = (
                    f'⚠️ **{p.name}** — {total} lines parsed, no AG guiding data.'
                )
            return

        active_panels = list(panels_check.value)
        fig = _build_figure(store, active_panels, x_start, x_end)
        plot_pane.object = fig
        _frame_ts.clear()
        _frame_ts.update({r.frame_id: r.t for r in store.snapshot('guide') if r.frame_id is not None})
        scatter_col_pane.object = _build_scatter_column(store, show_acq_frames.value, clamp_offsets=not full_offset_range.value)
        g = len(store.snapshot('guide'))
        v = len(store.snapshot('visit_changes'))
        d = len(store.snapshot('design_changes'))
        status_md.object = f'**Exposures:** {g} · **Visits:** {v} · **Designs:** {d}'

    except Exception as exc:
        import traceback
        plot_pane.object = go.Figure()
        scatter_col_pane.object = go.Figure()
        status_md.object = f'❌ `{type(exc).__name__}: {exc}`'
        traceback.print_exc()


# ── Tail mode ─────────────────────────────────────────────────────────────────

def _stop_follow():
    if _follow_cb[0] is not None:
        try:
            _follow_cb[0].stop()
        except Exception:
            pass
        _follow_cb[0] = None


def _init_tail():
    global _tail_records, _tail_state
    p = _log_path()
    if not p or not p.exists():
        return
    records = _parse_log(str(p))
    _tail_records = {k: list(v) for k, v in records.items()}
    _tail_state = {}
    with open(str(p), errors='replace') as f:
        f.seek(0, 2)
        _tail_pos[0] = f.tell()


def _tail_tick():
    p = _log_path()
    if not p or not p.exists():
        return
    try:
        new_recs, new_pos = _parse_from_pos(str(p), _tail_pos[0], _tail_state)
        _tail_pos[0] = new_pos
        if new_recs:
            for rtype, recs in new_recs.items():
                _tail_records.setdefault(rtype, []).extend(recs)
            if 'design_changes' in new_recs:
                _busy[0] = True
                design_select.options = _build_design_labels(
                    _tail_records.get('design_changes', [])
                )
                _busy[0] = False
            _render()
    except Exception as exc:
        status_md.object = f'❌ Tail error: `{exc}`'


# ── Widget callbacks ──────────────────────────────────────────────────────────

def _load_file():
    """Parse the selected file, refresh the design dropdown, and render.
    Called explicitly so directory changes always trigger a full reload even
    when file_select.value hasn't changed."""
    v = file_select.value
    if not v:
        plot_pane.object = go.Figure()
        status_md.object = '⚠️ No log file selected.'
        return
    p = Path(dir_input.value.strip()) / v
    if not p.exists():
        status_md.object = f'⚠️ Not found: `{p}`'
        return
    _busy[0] = True
    records = _parse_log(str(p))
    _refresh_design_list(records)
    _busy[0] = False
    _render()


@pn.depends(dir_input.param.value, watch=True)
def _on_dir(value):
    _stop_follow()
    follow_toggle.value = False
    _busy[0] = True          # suppress _on_file watcher during list refresh
    _refresh_file_list(value)
    _busy[0] = False
    _load_file()             # explicit reload regardless of whether value changed


@pn.depends(file_select.param.value, watch=True)
def _on_file(value):
    if _busy[0]:
        return               # _on_dir is driving; it will call _load_file()
    _stop_follow()
    follow_toggle.value = False
    _load_file()


@pn.depends(design_select.param.value, watch=True)
def _on_design(value):
    if not _busy[0]:
        _render()


@pn.depends(panels_check.param.value, watch=True)
def _on_panels(value):
    _render()


@pn.depends(show_acq_frames.param.value, watch=True)
def _on_show_acq(value):
    _render()


@pn.depends(full_offset_range.param.value, watch=True)
def _on_full_offset_range(value):
    _render()


@pn.depends(guide_coords.param.value, watch=True)
def _on_guide_coords(value):
    _render()


@pn.depends(tail_window_select.param.value, watch=True)
def _on_tail_window(value):
    if follow_toggle.value:
        _render()


@pn.depends(follow_toggle.param.value, watch=True)
def _on_follow(active):
    _stop_follow()
    if active:
        follow_toggle.name = '⏸ Following…'
        _init_tail()
        _tail_tick()
        _follow_cb[0] = pn.state.add_periodic_callback(
            _tail_tick, period=interval_input.value * 1000,
        )
    else:
        follow_toggle.name = '▶ Follow / Tail'
        _render()


@pn.depends(show_all_files.param.value, watch=True)
def _on_show_all(value):
    _stop_follow()
    follow_toggle.value = False
    _busy[0] = True
    _refresh_file_list(dir_input.value)
    _busy[0] = False
    _load_file()


# ── Layout ────────────────────────────────────────────────────────────────────

sidebar = pn.Column(
    pn.pane.Markdown('### Controls', margin=(5, 5, 0, 5)),
    dir_input,
    file_select,
    show_all_files,
    design_select,
    pn.layout.Divider(),
    pn.pane.Markdown('**Panels**', margin=(0, 5)),
    panels_check,
    show_acq_frames,
    full_offset_range,
    guide_coords,
    pn.layout.Divider(),
    pn.Row(follow_toggle, interval_input),
    tail_window_select,
    pn.layout.Divider(),
    status_md,
    width=310,
    sizing_mode='fixed',
)

app = pn.Row(
    sidebar,
    pn.Column(plot_pane, sizing_mode='stretch_both'),
    scatter_col_pane,
    sizing_mode='stretch_both',
)

app.servable()

# Initial population (runs in notebook; panel serve calls servable() instead)
_refresh_file_list(dir_input.value)
